# 15 · Model 1 profile pilot with the new teacher (Qwen3.5-9B, thinking off)
Owner: Disha. Re-runs the 200-user pilot of notebook 14 with **Qwen3.5-9B** in place of Qwen2.5-7B, before the full run
of all 56,017 users. Same 200 users (seed 42), same prompt (v2, identical hash), so the two pilots compare like for like.

### What this pilot must show
1. **Thinking is really off**: no `<think>` text in any profile (checked automatically).
2. **Speed on this GPU**: seconds per profile, and the estimate for all 56,017 users.
3. **Quality**: automatic flags, a side-by-side with the Qwen2.5-7B pilot, and **20 profiles read by hand**.
4. **The pinned model commit**, to send to Abhinita for the prototype endpoint.

### How to run
* **Google Colab:** Runtime → Change runtime type → **GPU** (A100 or L4; a T4 also works, the model then loads in 4-bit).
  Runtime → Run all, and allow Drive access.
* **Ashley's RTX 4090 (local Jupyter):** in the settings cell set `DATA_DIR` and `OUT_ROOT` to where the shared Drive
  folder is synced on that computer (Google Drive for desktop), then Run all.
* The first cell installs pinned library versions. If it asks you to **restart**, restart the runtime/kernel and Run all again.
* The pilot saves every 50 users. If it stops, Run all again: finished parts are skipped.

### What it reads (frozen data, never changed; hashes checked against the frozen manifest)
`sequences.parquet` (`user_idx`, `train_items`, `n_train` only), `item_text.parquet` (Title, Brand, Category, Price
parts only), `interactions.parquet` (only rows with `split == 'train'`, for the user's own star ratings).
Validation and test items are never read.

### What it writes: `<OUT_ROOT>/pilot_v2_qwen35_9b/` on the shared Drive (never to Git)
| File | Contents |
|---|---|
| `user_profiles.parquet` | `user_idx`, `profile_text`, `teacher_model`, `prompt_version`, `n_train_items_used`, `input_sha256` |
| `profiles_minilm.npz`, `profiles_e5_large.npz` | agreed profile file format: `user_idx`, `embeddings`, `source="llm_profile"`, `train_items_only=True` |
| `profiles_meta.json` | model and commit, how it was loaded, the exact prompt, encoders and commits, input file hashes, library versions |
| `pilot_report.json` | failures, thinking check, length, speed, estimate for all users, flag counts |
| `review_sample.csv` | 20 profiles next to their products, **to read by hand** |
| `comparison_with_pilot_v2.csv` | the same users' Qwen2.5-7B and Qwen3.5-9B profiles side by side |
| `profile_flags.csv`, `consistency_check.csv`, `failed_users.json`, `checksums.json`, `chunks/` | checks and progress |

The earlier Qwen2.5-7B pilot in `pilot_v2/` is only read, never written.

In [1]:
# ============================== SETTINGS (edit here) ==============================
# Shared Drive locations. Colab: the paths below (same as the Qwen2.5-7B pilot).
# Ashley's 4090: the synced Drive folder on that computer, e.g. '/home/<user>/GoogleDrive/298A/...' or r'G:\My Drive\298A\...'.
DATA_DIR = '/content/drive/MyDrive/298A/04_Data/NewDataRunFiles/processed'   # frozen data (read only)
OUT_ROOT = '/content/drive/MyDrive/298A/model1_profiles'                     # this pilot writes to OUT_ROOT/RUN_NAME

TEACHER_MODEL = 'Qwen/Qwen3.5-9B'
TEACHER_REVISION = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'   # pinned commit (Hugging Face main on Nov 22, 2026)
ENABLE_THINKING = False                                         # must stay False: Qwen3.5 thinks by default
MAX_NEW_TOKENS = 160
BATCH_SIZE = 8                                                  # lowered automatically if the GPU runs out of memory

ENCODERS = ['minilm', 'e5_large']
ENCODER_REVISIONS = {'minilm': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41',     # same commit as Abhinita's item vectors
                     'e5_large': '3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3'}

PROMPT_VERSION = 'v2'                  # unchanged from the Qwen2.5-7B pilot, so the comparison is like for like
RUN_NAME = 'pilot_v2_qwen35_9b'        # output folder; never reuse a folder made with another model or prompt
COMPARE_WITH = 'pilot_v2'              # the Qwen2.5-7B pilot folder under OUT_ROOT (read only)
MAX_ITEMS = 50                         # most recent 50 training items per user
CHUNK_SIZE = 50                        # users per saved part (resume after a crash)
SEED = 42
PILOT_STRATA = [(3, 4, 100), (5, 9, 60), (10, None, 40)]   # (min n_train, max n_train, users) -> the same 200 users
CONSISTENCY_USERS = 8                  # users regenerated one at a time to check that batching does not change outputs
N_USERS_FULL = 56017
COST_PER_GPU_HOUR_USD = None           # optional, for the cost estimate
# ==================================================================================

In [2]:
# Install pinned library versions (Transformers 5.x is needed for Qwen3.5).
import sys, subprocess, importlib.metadata as md
PINNED = {'transformers': '5.24.0', 'sentence-transformers': '6.3.0'}
def _ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None
missing = {n: v for n, v in PINNED.items() if _ver(n) != v}
if missing:
    print('Installing', missing, '...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers[torch]==5.24.0',
                    'sentence-transformers==6.3.0'], check=True)
    if any(m in sys.modules for m in ('transformers', 'sentence_transformers', 'huggingface_hub', 'tokenizers')):
        raise RuntimeError('Installed new library versions. RESTART the runtime/kernel now, then Run all again.')
print({n: _ver(n) for n in ('torch', 'transformers', 'sentence-transformers', 'huggingface-hub', 'accelerate')})

Installing {'transformers': '5.24.0', 'sentence-transformers': '6.3.0'} ...
{'torch': '2.14.1+cu130', 'transformers': '5.24.0', 'sentence-transformers': '6.3.0', 'huggingface-hub': '1.33.0', 'accelerate': '1.15.0'}


## 1. Locations: shared Drive, output folder, safety checks

In [3]:
import os, json, re, time, hashlib, datetime, random, glob
from pathlib import Path
import numpy as np
import pandas as pd

for k in ('DATA_DIR', 'OUT_ROOT', 'TEACHER_MODEL', 'RUN_NAME'):          # environment overrides (local testing only)
    if os.environ.get(f'PROFILES_{k}'):
        globals()[k] = os.environ[f'PROFILES_{k}']
if os.environ.get('PROFILES_TEACHER_REVISION'):
    TEACHER_REVISION = os.environ['PROFILES_TEACHER_REVISION']

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR, OUT_ROOT = Path(DATA_DIR), Path(OUT_ROOT)
if not (DATA_DIR / 'sequences.parquet').exists():
    # The shared folder can appear under MyDrive (a shortcut) or under Shareddrives; look for it before giving up.
    found = sorted({str(Path(p).parent) for pat in ('/content/drive/MyDrive/**/NewDataRunFiles/processed/sequences.parquet',
                                                     '/content/drive/Shareddrives/**/NewDataRunFiles/processed/sequences.parquet')
                    for p in glob.glob(pat, recursive=True)}) if IN_COLAB else []
    if len(found) == 1:
        DATA_DIR = Path(found[0])
        OUT_ROOT = DATA_DIR.parents[2] / 'model1_profiles'      # .../298A/04_Data/NewDataRunFiles/processed -> .../298A
        print('NOTE: frozen data found at another location; using it and the matching output folder.')
    else:
        raise FileNotFoundError(f'sequences.parquet not found in {DATA_DIR}. Set DATA_DIR in the settings cell to the shared '
                                f'Drive folder with the frozen data. Candidates found: {found or "none"}')
assert (OUT_ROOT / COMPARE_WITH).resolve() != (OUT_ROOT / RUN_NAME).resolve(), 'never write into the earlier pilot folder'
OUT_DIR = OUT_ROOT / RUN_NAME
CHUNK_DIR = OUT_DIR / 'chunks'
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
_probe = OUT_DIR / '.write_test'
_probe.write_text('ok'); _probe.unlink()                               # fail now, not after an hour of generation
random.seed(SEED); np.random.seed(SEED)
print('Frozen data :', DATA_DIR)
print('Output      :', OUT_DIR, '(writable)')
print('Compare with:', OUT_ROOT / COMPARE_WITH, '(exists)' if (OUT_ROOT / COMPARE_WITH).exists() else '(not found: side-by-side skipped)')

Mounted at /content/drive
Frozen data : /content/drive/MyDrive/298A/04_Data/NewDataRunFiles/processed
Output      : /content/drive/MyDrive/298A/model1_profiles/pilot_v2_qwen35_9b (writable)
Compare with: /content/drive/MyDrive/298A/model1_profiles/pilot_v2 (exists)


## 2. Check the frozen inputs against the manifest
The three files must have the SHA-256 recorded in the frozen manifest (`docs/data_run/manifest.json`, manifest
SHA-256 `e4b7801c…`). A different data version stops the run here.

In [4]:
EXPECTED_SHA256 = {
    'sequences.parquet': '0a55f2c375c06594b322d1e23dcd49d975ac7f8e1f7db12e4cc8bea59128186a',
    'item_text.parquet': '39dbd1ac817df3383def7c677fca830aaf36cfe470e1c0612c543486e6b8909d',
    'interactions.parquet': '298cab581ac90b44504d512b6c706f4616ea93affa8e601be58158a8252c4905',
}
EXPECTED_MANIFEST_SHA256 = 'e4b7801c0d9ea27d0b39a6af253ac154d6f4a7181d8a7c1dc31d651a050af0e5'
if os.environ.get('PROFILES_EXPECTED_HASHES_JSON'):                    # local testing with synthetic data only
    EXPECTED_SHA256 = json.loads(Path(os.environ['PROFILES_EXPECTED_HASHES_JSON']).read_text())
    EXPECTED_MANIFEST_SHA256 = None

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

INPUT_SHA256 = {name: sha256_file(DATA_DIR / name) for name in EXPECTED_SHA256}
bad = {n: h for n, h in INPUT_SHA256.items() if h != EXPECTED_SHA256[n]}
assert not bad, f'input files differ from the frozen manifest: {bad}'
if EXPECTED_MANIFEST_SHA256 and (DATA_DIR / 'manifest.json').exists():
    assert sha256_file(DATA_DIR / 'manifest.json') == EXPECTED_MANIFEST_SHA256, 'manifest.json is not the frozen one'
print('PASS input files match the frozen manifest:', {n: h[:12] for n, h in INPUT_SHA256.items()})

PASS input files match the frozen manifest: {'sequences.parquet': '0a55f2c375c0', 'item_text.parquet': '39dbd1ac817d', 'interactions.parquet': '298cab581ac9'}


## 3. Load the training histories and the product text
Only `user_idx`, `train_items` and `n_train` are loaded, so validation and test items cannot reach a prompt.

In [5]:
seq = pd.read_parquet(DATA_DIR / 'sequences.parquet', columns=['user_idx', 'train_items', 'n_train'])
seq['train_items'] = seq.train_items.apply(lambda a: [int(i) for i in a])
assert seq.user_idx.is_unique and (seq.train_items.str.len() == seq.n_train).all()

items = pd.read_parquet(DATA_DIR / 'item_text.parquet', columns=['item_idx', 'item_text'])
assert items.item_idx.is_unique and items.item_text.str.len().gt(0).all()

FIELDS_KEPT = ('Title', 'Brand', 'Category', 'Price')
_FIELD_SPLIT = re.compile(r' \| (?=(?:Title|Brand|Category|Features|Description|Price|Reviews): )')

def short_item_line(item_text, title_words=30):
    # keep the Title, Brand, Category and Price parts of item_text (metadata only)
    parts = {}
    for part in _FIELD_SPLIT.split(item_text):
        key, _, value = part.partition(': ')
        if key in FIELDS_KEPT and value.strip() and value.strip() != '...':
            value = value.strip()
            if key == 'Title':
                words = value.split()
                value = ' '.join(words[:title_words]) + (' ...' if len(words) > title_words else '')
            parts[key] = value
    return ' | '.join(f'{k}: {parts[k]}' for k in FIELDS_KEPT if k in parts)

inter = pd.read_parquet(DATA_DIR / 'interactions.parquet', columns=['user_idx', 'item_idx', 'rating', 'split'])
assert set(inter.split.unique()) <= {'train', 'val', 'test'}
train_r = inter[inter.split == 'train']                                  # validation and test rows are never used
RATING = dict(zip(zip(train_r.user_idx.astype(int), train_r.item_idx.astype(int)), train_r.rating.astype(float)))
assert len(RATING) == len(train_r), 'expected one training rating per user-item pair'
print(f'Training ratings loaded: {len(RATING):,} (validation and test rows ignored: {len(inter) - len(train_r):,})')
del inter, train_r

ITEM_LINE = dict(zip(items.item_idx, items.item_text.map(short_item_line)))
assert all(ITEM_LINE.values()), 'every item needs at least a title'
print(f'{len(seq):,} users, {len(items):,} items')

Training ratings loaded: 389,365 (validation and test rows ignored: 112,034)
56,017 users, 25,903 items


## 4. Choose the pilot users (the same 200 as the Qwen2.5-7B pilot)
200 users drawn with seed 42 by training-history length: 100 with 3 to 4 items, 60 with 5 to 9, 40 with 10 or more.
Each user also gets `input_sha256`, a hash of exactly the items and ratings in their prompt, so the audit can later
re-derive every input from the frozen data without trusting the `train_items_only` flag.

In [6]:
rng = np.random.default_rng(SEED)
picks = []
for lo, hi, n in PILOT_STRATA:
    pool = seq[(seq.n_train >= lo) & ((seq.n_train <= hi) if hi is not None else True)].user_idx.sort_values().to_numpy()
    assert len(pool) >= n, (lo, hi, len(pool))
    picks.append(np.sort(rng.choice(pool, size=n, replace=False)))
users = seq.set_index('user_idx').loc[np.concatenate(picks)].reset_index()
SELECTION_RULE = (f'pilot: seed {SEED}, numpy default_rng, without replacement within strata of n_train: '
                  + ', '.join(f"{n} users with n_train {lo}-{hi if hi is not None else 'max'}" for lo, hi, n in PILOT_STRATA))

users['items_used'] = users.train_items.apply(lambda l: l[-MAX_ITEMS:])
users['n_train_items_used'] = users.items_used.str.len()
users['ratings_used'] = [[RATING.get((int(u), i)) for i in l] for u, l in zip(users.user_idx, users.items_used)]
assert sum(r is None for l in users.ratings_used for r in l) == 0, 'a used item has no training rating'
assert (users.n_train_items_used > 0).all()

def input_hash(item_ids, ratings):
    return hashlib.sha256(json.dumps({'items': [int(i) for i in item_ids], 'ratings': [float(r) for r in ratings]},
                                     separators=(',', ':')).encode()).hexdigest()
users['input_sha256'] = [input_hash(i, r) for i, r in zip(users.items_used, users.ratings_used)]
print('PASS every item in a prompt has its own training rating (split == train)')
print(SELECTION_RULE)
print(f'{len(users)} users to profile;', users.n_train_items_used.describe().round(1).to_dict())

PASS every item in a prompt has its own training rating (split == train)
pilot: seed 42, numpy default_rng, without replacement within strata of n_train: 100 users with n_train 3-4, 60 users with n_train 5-9, 40 users with n_train 10-max
200 users to profile; {'count': 200.0, 'mean': 7.9, 'std': 9.2, 'min': 3.0, '25%': 3.0, '50%': 4.5, '75%': 7.2, 'max': 50.0}


## 5. The prompt (v2, unchanged)
Copied verbatim from notebook 14, so `prompt_sha256` equals the Qwen2.5-7B pilot's. Products are listed oldest to
newest, each with the user's own training star rating. No user id, review, validation or test item is included.

In [7]:
PROMPT_TEMPLATE = '''You are helping a skin-care shop understand what one of its shoppers likes.

Below are the products this shopper rated, from oldest to newest. Each line is the product's catalog information (with a price band only when the catalog has one), followed by the shopper's own star rating (1 to 5 stars).

{item_lines}

Write a short profile of this shopper's skin-care interests in 2 to 4 sentences (at most 100 words).

How to read the ratings:
- 4 or 5 stars: the shopper liked the product.
- 3 stars: neutral. Do not describe it as liked or disliked.
- 1 or 2 stars: the shopper disliked the product. If there are any, name the disliked product types or brands.

What to describe:
- The product types, categories and brands of the liked products.
- Price level, only from the prices actually shown. Many products have no price, so never say "all" or "always" about prices unless every product shows one. If few prices are shown, leave price out.
- Needs or benefits only when the product titles state them.

Rules:
- Use only what the product list shows. Do not invent products, brands, prices, needs or benefits.
- Never say the shopper is not interested in, avoids or does not care about something that appears in the list.
- Do not use vague praise such as "high-quality" or "effective". Say "natural" or "organic" only if most of the listed titles say so.
- Describe products and interests only. Do not guess or state any health condition, age, gender, ethnicity or other personal identity.
- Refer to the person as "this shopper". Do not use lists, headings or any ids.
- Reply with the profile text only.'''
PROMPT_SHA = hashlib.sha256(PROMPT_TEMPLATE.encode()).hexdigest()

def build_prompt(item_ids, ratings):
    return PROMPT_TEMPLATE.format(item_lines='\n'.join(
        f'{k}. {ITEM_LINE[i]} | Rating: {r:g}/5' for k, (i, r) in enumerate(zip(item_ids, ratings), 1)))

users['prompt'] = [build_prompt(i, r) for i, r in zip(users.items_used, users.ratings_used)]
assert not users.prompt.str.contains(r'(?:Features|Description|Reviews): ').any(), 'only Title/Brand/Category/Price allowed'
old_meta = OUT_ROOT / COMPARE_WITH / 'profiles_meta.json'
if old_meta.exists():
    old_sha = json.loads(old_meta.read_text()).get('prompt_sha256')
    assert old_sha == PROMPT_SHA, f'prompt differs from the {COMPARE_WITH} pilot ({old_sha[:12]} vs {PROMPT_SHA[:12]})'
    print(f'PASS prompt is identical to the {COMPARE_WITH} pilot')
print('prompt_sha256:', PROMPT_SHA)
ex = users.sort_values('n_train_items_used').iloc[0]
print(f'Example prompt ({ex.n_train_items_used} items):\n' + '-' * 60 + '\n' + ex.prompt)

PASS prompt is identical to the pilot_v2 pilot
prompt_sha256: a18472f1aefb1a5f3b6397d20d6dc2335a382a6b69c681b908e3e23ba97e9727
Example prompt (3 items):
------------------------------------------------------------
You are helping a skin-care shop understand what one of its shoppers likes.

Below are the products this shopper rated, from oldest to newest. Each line is the product's catalog information (with a price band only when the catalog has one), followed by the shopper's own star rating (1 to 5 stars).

1. Title: Medline Remedy Phytoplex Nourishing Skin Cream, Skin Moisturizer, Paraben Free Body Lotion, 16 Fl Oz, 2 Pack | Brand: Remedy Dermatology Series | Category: Skin Care > Body > Moisturizers > Creams | Price: $10-25 | Rating: 5/5
2. Title: Neutrogena Cleansing Fragrance Free Makeup Remover Face Wipes, Cleansing Facial Towelettes for Waterproof Makeup, Alcohol-Free, Unscented, 100% Plant-Based Fibers, Twin Pack, 2 x 25 ct | Brand: Neutrogena | Category: Skin Care > Face > Cle

## 6. Load the teacher (Qwen3.5-9B, text only, pinned commit, thinking off)
* The checkpoint is multimodal; `AutoModelForCausalLM` loads only its language model (`Qwen3_5ForCausalLM`), so the
  vision part takes no GPU memory.
* 20 GB or more of GPU memory (RTX 4090, L4, A100): full bf16, about 19 GB. Less (T4): 4-bit NF4, noted in the report.
* Thinking: the chat template is rendered once with `enable_thinking=False` and must end with an empty
  `<think></think>` block. Otherwise the run stops here.
* Generation is greedy (no sampling), at most 160 new tokens, so reruns give the same text.

In [8]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.manual_seed(SEED)
ALLOW_CPU = os.environ.get('PROFILES_ALLOW_CPU') == '1'                 # local testing only
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError('No GPU found. Colab: Runtime > Change runtime type > GPU. Local: check the CUDA install.')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
print(f'Device: {GPU_NAME} ({VRAM_GB:.0f} GB)')

IS_LOCAL_MODEL = Path(TEACHER_MODEL).exists()
CURRENT_MAIN = None
if not IS_LOCAL_MODEL:
    from huggingface_hub import HfApi
    api = HfApi()
    assert api.model_info(TEACHER_MODEL, revision=TEACHER_REVISION).sha == TEACHER_REVISION, 'pinned commit not found'
    CURRENT_MAIN = api.model_info(TEACHER_MODEL).sha
    if CURRENT_MAIN != TEACHER_REVISION:
        print(f'NOTE: {TEACHER_MODEL} main is now {CURRENT_MAIN}; this run stays on the pinned {TEACHER_REVISION}.')
TEACHER_REV = 'local' if IS_LOCAL_MODEL else TEACHER_REVISION
_rev = None if IS_LOCAL_MODEL else TEACHER_REVISION

tok = AutoTokenizer.from_pretrained(TEACHER_MODEL, revision=_rev)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def chat_text(prompt):
    return tok.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True,
                                   enable_thinking=ENABLE_THINKING)
GEN_PROMPT_TAIL = chat_text('x')[-40:]
assert ENABLE_THINKING is False and chat_text('x').endswith('<think>\n\n</think>\n\n'), \
    f'thinking is NOT switched off by the chat template; generation prompt ends with {GEN_PROMPT_TAIL!r}'
print('PASS thinking is off: generation prompt ends with', repr(GEN_PROMPT_TAIL[-30:]))

kwargs = {}
if torch.cuda.is_available() and VRAM_GB < 20:
    import importlib.util
    if importlib.util.find_spec('bitsandbytes') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'], check=True)
    from transformers import BitsAndBytesConfig
    kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16)
    LOAD_NOTE = f'4-bit NF4 (GPU has {VRAM_GB:.0f} GB)'
elif torch.cuda.is_available():
    kwargs['dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    LOAD_NOTE = str(kwargs['dtype'])
else:
    kwargs['dtype'] = torch.float32
    LOAD_NOTE = 'float32 on CPU (test only)'
model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, revision=_rev,
                                             device_map='auto' if torch.cuda.is_available() else None, **kwargs)
model.eval()
TEACHER_CLASS = type(model).__name__
assert TEACHER_CLASS == 'Qwen3_5ForCausalLM', f'unexpected model class {TEACHER_CLASS}'
EOS_IDS = model.generation_config.eos_token_id
EOS_IDS = set(EOS_IDS if isinstance(EOS_IDS, (list, tuple)) else [EOS_IDS]) | {tok.eos_token_id}
if torch.cuda.is_available():
    print(f'GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
TEACHER_TAG = f'{TEACHER_MODEL if not IS_LOCAL_MODEL else "local:" + Path(TEACHER_MODEL).name}@{TEACHER_REV}'
print('Teacher:', TEACHER_TAG, '| class:', TEACHER_CLASS, '| loaded as:', LOAD_NOTE)

def generate_batch(prompts):
    # returns [(text, had_think_tag, hit_max_tokens)] in input order
    enc = tok([chat_text(p) for p in prompts], return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = out[:, enc['input_ids'].shape[1]:]
    texts = tok.batch_decode(gen, skip_special_tokens=True)
    raw = tok.batch_decode(gen, skip_special_tokens=False)               # keeps <think> even if it is a special token
    had_think = ['<think>' in r or '</think>' in r for r in raw]
    hit_max = [not any(int(t) in EOS_IDS for t in row) for row in gen]
    return list(zip(texts, had_think, hit_max))

Device: NVIDIA L4 (24 GB)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

GPU memory after loading: 17.9 GB
Teacher: Qwen/Qwen3.5-9B@c202236235762e1c871ad0ccb60c8ee5ba337b9a | class: Qwen3_5ForCausalLM | loaded as: torch.bfloat16


## 7. Generate the profiles (saved every 50 users)
Each part is written to `chunks/` as soon as it finishes (written to a temporary name first, then renamed, so no
half-written part). A rerun skips parts made with the same users, inputs, prompt, model and thinking setting.
A user whose output is empty is recorded with the reason, never dropped.

In [9]:
THINK_RE = re.compile(r'<think>.*?</think>', re.S)

def clean(raw):
    text = THINK_RE.sub(' ', str(raw)).replace('<think>', ' ').replace('</think>', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^(profile|shopper profile)\s*:\s*', '', text, flags=re.I).strip().strip('"').strip()
    return text

def run_users(df):
    df = df.assign(_len=df.prompt.str.len()).sort_values('_len', ascending=False)   # similar lengths per batch
    rows, bs, i = [], BATCH_SIZE, 0
    while i < len(df):
        batch = df.iloc[i:i + bs]
        t0, err = time.time(), ''
        try:
            outs = generate_batch(batch.prompt.tolist())
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs > 1:
                bs = max(1, bs // 2); print(f'  out of GPU memory -> batch size {bs}'); continue
            outs, err = [(None, False, False)] * len(batch), 'out of GPU memory'
        except Exception as e:                                           # record, never skip silently
            outs, err = [(None, False, False)] * len(batch), f'{type(e).__name__}: {e}'[:300]
        secs = (time.time() - t0) / len(batch)
        for (_, u), (raw, had_think, hit_max) in zip(batch.iterrows(), outs):
            text = clean(raw) if raw is not None else ''
            rows.append(dict(user_idx=int(u.user_idx), profile_text=text, status='ok' if text else 'failed',
                             reason='' if text else (err or 'empty output'),
                             raw_had_think_tag=bool(had_think) or (raw is not None and '<think>' in raw),
                             hit_max_tokens=bool(hit_max),
                             n_train_items_used=int(u.n_train_items_used), input_sha256=u.input_sha256,
                             seconds=secs, batch_size=len(batch), prompt_sha=PROMPT_SHA, teacher=TEACHER_TAG,
                             enable_thinking=ENABLE_THINKING))
        i += len(batch)
    return pd.DataFrame(rows)

chunks = [users.iloc[s:s + CHUNK_SIZE] for s in range(0, len(users), CHUNK_SIZE)]
STOP_AFTER = int(os.environ['PROFILES_STOP_AFTER_CHUNKS']) if os.environ.get('PROFILES_STOP_AFTER_CHUNKS') else None
t_start, done_now = time.time(), 0
for c, part in enumerate(chunks):
    path = CHUNK_DIR / f'chunk_{c:05d}.parquet'
    if path.exists():
        prev = pd.read_parquet(path)
        same = (sorted(prev.user_idx) == sorted(part.user_idx)
                and set(zip(prev.user_idx, prev.input_sha256)) == set(zip(part.user_idx, part.input_sha256))
                and prev.prompt_sha.eq(PROMPT_SHA).all() and prev.teacher.eq(TEACHER_TAG).all()
                and prev.enable_thinking.eq(ENABLE_THINKING).all())
        if same:
            print(f'part {c + 1}/{len(chunks)}: already done, skipped')
            continue
        print(f'part {c + 1}/{len(chunks)}: saved part was made with another setup -> regenerating')
    if STOP_AFTER is not None and done_now >= STOP_AFTER:
        raise SystemExit(f'test hook: simulated stop after {done_now} part(s)')
    res = run_users(part)
    tmp = path.with_suffix('.tmp')
    res.to_parquet(tmp, index=False)
    os.replace(tmp, path)
    done_now += 1
    print(f'part {c + 1}/{len(chunks)}: {len(res)} users, {(res.status == "failed").sum()} failed, '
          f'{(time.time() - t_start) / 60:.1f} min so far')

res = pd.concat([pd.read_parquet(CHUNK_DIR / f'chunk_{c:05d}.parquet') for c in range(len(chunks))], ignore_index=True)
assert res.user_idx.is_unique and set(res.user_idx) == set(users.user_idx), 'every pilot user must have a result'
print(f'{len(res)} users: {(res.status == "ok").sum()} profiles, {(res.status == "failed").sum()} failed, '
      f'{int(res.raw_had_think_tag.sum())} with thinking text, {int(res.hit_max_tokens.sum())} cut at {MAX_NEW_TOKENS} tokens')

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


part 1/4: 50 users, 0 failed, 1.3 min so far
part 2/4: 50 users, 0 failed, 2.6 min so far
part 3/4: 50 users, 0 failed, 4.1 min so far
  out of GPU memory -> batch size 4
  out of GPU memory -> batch size 2
part 4/4: 50 users, 0 failed, 8.7 min so far
200 users: 200 profiles, 0 failed, 0 with thinking text, 0 cut at 160 tokens


## 8. Save the profiles and the failed users

In [10]:
ok = res[res.status == 'ok'].sort_values('user_idx')
profiles = pd.DataFrame({
    'user_idx': ok.user_idx.astype('int64'),
    'profile_text': ok.profile_text.astype(str),
    'teacher_model': TEACHER_TAG,
    'prompt_version': PROMPT_VERSION,
    'n_train_items_used': ok.n_train_items_used.astype('int64'),
    'input_sha256': ok.input_sha256.astype(str),
}).reset_index(drop=True)
profiles.to_parquet(OUT_DIR / 'user_profiles.parquet', index=False)
failed = [dict(user_idx=int(r.user_idx), reason=r.reason) for r in res[res.status == 'failed'].itertuples()]
(OUT_DIR / 'failed_users.json').write_text(json.dumps(failed, indent=1))
assert len(profiles) + len(failed) == len(users), 'every user is either profiled or listed as failed'
print(f'Saved user_profiles.parquet ({len(profiles)} rows) and failed_users.json ({len(failed)} users)')

Saved user_profiles.parquet (200 rows) and failed_users.json (0 users)


## 9. Does batching change the outputs?
Qwen3.5 mixes linear-attention and attention layers. A few users are regenerated one at a time (no padding) and
compared with their batched output. Identical or near-identical text is expected; tiny differences can come from
GPU arithmetic. Clearly different profiles would point to a padding problem: then use batch size 1 or vLLM for the
full run.

In [11]:
from difflib import SequenceMatcher
chk = profiles.sample(min(CONSISTENCY_USERS, len(profiles)), random_state=SEED).merge(users[['user_idx', 'prompt']], on='user_idx')
single = [clean(generate_batch([p])[0][0]) for p in chk.prompt]
consistency = pd.DataFrame({'user_idx': chk.user_idx, 'batched': chk.profile_text, 'single': single})
consistency['identical'] = consistency.batched == consistency.single
consistency['similarity'] = [round(SequenceMatcher(None, a, b).ratio(), 3) for a, b in zip(consistency.batched, consistency.single)]
consistency.to_csv(OUT_DIR / 'consistency_check.csv', index=False)
CONSISTENCY = {'users': int(len(consistency)), 'identical': int(consistency.identical.sum()),
               'min_similarity': float(consistency.similarity.min())}
print(CONSISTENCY)
if CONSISTENCY['min_similarity'] < 0.8:
    print('WARNING: batched and one-at-a-time outputs differ clearly. Read consistency_check.csv before the full run.')

{'users': 8, 'identical': 3, 'min_similarity': 0.626}


## 10. Checks to read by hand
The flags only help the reading; they reject nothing. **Read the 20 profiles in `review_sample.csv`** and check:
(1) nothing that does not follow from the listed products, (2) no sensitive inference (health, age, gender,
ethnicity, identity), (3) sensible length, (4) no reasoning text or meta comments such as "Let me think".

In [12]:
words = profiles.profile_text.str.split().str.len()
SENSITIVE = re.compile(r"\b(she|he|her|his|woman|women|man|men|female|male|girl|boy|teen\w*|age[ds]?|old|young|"
                       r"years|mother|mom|pregnan\w*|baby|wife|husband|asian|black|white|latin\w*|african|"
                       r"diabet\w*|eczema|psoriasis|rosacea|cancer|disease|condition|allerg\w*|acne)\b", re.I)
per_user = users.set_index('user_idx')
item_blob = {u: ' '.join(ITEM_LINE[i] for i in l).lower() for u, l in zip(users.user_idx, users.items_used)}

def sensitive_terms(uid, text):
    # terms from the list that are NOT simply copied from the user's own product text (e.g. the brand "Jack Black")
    found = {m.lower() for m in SENSITIVE.findall(text)}
    return ', '.join(sorted(t for t in found if not re.search(rf'\b{re.escape(t)}\b', item_blob[uid])))

LOW_MENTION = re.compile(r"dislik|low(er)?[- ]rat|poor|[12][- ]star|[12]/5|did not like|didn't like|less favou?r|negative", re.I)
has_low = per_user.ratings_used.apply(lambda rr: any(r <= 2 for r in rr))
res_ok = res.set_index('user_idx')

def make_flags(p):
    return pd.DataFrame({
        'user_idx': p.user_idx,
        'words': p.profile_text.str.split().str.len(),
        'over_120_words': p.profile_text.str.split().str.len() > 120,
        'possible_sensitive_terms': [sensitive_terms(u, t) for u, t in zip(p.user_idx, p.profile_text)],
        'mentions_digits_like_ids': p.profile_text.str.contains(r'\b\d{4,}\b'),
        'says_all_about_price': p.profile_text.str.contains(r"\b(?:all|every|always)\b[^.]{0,90}(?:\$|price)", case=False),
        'claims_absence': p.profile_text.str.contains(r"not (?:a )?(?:major )?concern|no (?:apparent )?interest|not interested|avoids?\b|does not care", case=False),
        'vague_praise': p.profile_text.str.contains(r"high[- ]quality|\beffective\b", case=False),
        'dislikes_not_mentioned': [bool(has_low[u]) and not LOW_MENTION.search(t) for u, t in zip(p.user_idx, p.profile_text)],
        'meta_or_reasoning_text': p.profile_text.str.contains(r"\blet me\b|\bokay,|\bfirst,? i\b|\bthe user\b|\bi (?:need|will|should)\b", case=False),
    }).reset_index(drop=True)

FLAG_COLS = ['over_120_words', 'mentions_digits_like_ids', 'says_all_about_price', 'claims_absence', 'vague_praise',
             'dislikes_not_mentioned', 'meta_or_reasoning_text']
flags = make_flags(profiles)
flags['raw_had_think_tag'] = flags.user_idx.map(res_ok.raw_had_think_tag).astype(bool)
flags['hit_max_tokens'] = flags.user_idx.map(res_ok.hit_max_tokens).astype(bool)
flags.to_csv(OUT_DIR / 'profile_flags.csv', index=False)
FLAG_COUNTS = {**{c: int(flags[c].sum()) for c in FLAG_COLS + ['raw_had_think_tag', 'hit_max_tokens']},
               'possible_sensitive_terms': int((flags.possible_sensitive_terms != '').sum())}
print(f'words per profile: mean {words.mean():.1f}, median {words.median():.0f}, max {words.max()}')
print('profiles flagged:', FLAG_COUNTS)

sample = profiles.sample(min(20, len(profiles)), random_state=SEED)
sample = sample.merge(users[['user_idx', 'items_used', 'ratings_used']], on='user_idx')
sample['products'] = [' || '.join(f'{ITEM_LINE[i]} | Rating: {r:g}/5' for i, r in zip(l, rr))
                      for l, rr in zip(sample.items_used, sample.ratings_used)]
sample[['user_idx', 'n_train_items_used', 'profile_text', 'products']].to_csv(OUT_DIR / 'review_sample.csv', index=False)
for r in sample.head(5).itertuples():
    print(f'\n--- user {r.user_idx} ({r.n_train_items_used} items) ---\n{r.profile_text}')

words per profile: mean 63.4, median 62, max 98
profiles flagged: {'over_120_words': 0, 'mentions_digits_like_ids': 2, 'says_all_about_price': 9, 'claims_absence': 0, 'vague_praise': 0, 'dislikes_not_mentioned': 6, 'meta_or_reasoning_text': 0, 'raw_had_think_tag': 0, 'hit_max_tokens': 0, 'possible_sensitive_terms': 1}

--- user 27470 (5 items) ---
This shopper favors gentle, soothing cleansers and toners, specifically liking Dove, Aveeno, Pond's, and Dickinson's brands. They show a strong preference for products designed for sensitive skin or those with calming ingredients like oat and witch hazel. While they enjoy a budget-friendly pore scrub, their top-rated items are body washes and face cleansers that offer hypoallergenic or natural benefits.

--- user 5449 (3 items) ---
This shopper favors self-tanners and lip balms, giving top ratings to a Fake Bake body self-tanner and an Artnaturals organic lip balm gift set. While they rated a Mario Badescu pore cleansing product neutrally, th

## 11. Side by side with the Qwen2.5-7B pilot
Same users, same prompt; only the teacher differs. `comparison_with_<pilot>.csv` holds both profiles for every user,
and the table compares the flag counts (computed the same way for both).

In [13]:
COMPARISON = None
old_path = OUT_ROOT / COMPARE_WITH / 'user_profiles.parquet'
if old_path.exists():
    old = pd.read_parquet(old_path)
    both = profiles[['user_idx', 'profile_text']].merge(old[['user_idx', 'profile_text', 'teacher_model']],
                                                        on='user_idx', suffixes=('_new', '_old'))
    both = both.rename(columns={'profile_text_new': 'qwen35_9b', 'profile_text_old': 'previous', 'teacher_model': 'previous_teacher'})
    both.to_csv(OUT_DIR / f'comparison_with_{COMPARE_WITH}.csv', index=False)
    old_flags = make_flags(old[old.user_idx.isin(both.user_idx)].reset_index(drop=True))
    new_flags = flags[flags.user_idx.isin(both.user_idx)]
    table = pd.DataFrame({'previous_pilot': {**{c: float(old_flags[c].sum()) for c in FLAG_COLS},
                                             'mean_words': round(float(old_flags.words.mean()), 1)},
                          'qwen35_9b': {**{c: float(new_flags[c].sum()) for c in FLAG_COLS},
                                        'mean_words': round(float(new_flags.words.mean()), 1)}})
    COMPARISON = {'users_in_both': int(len(both)), 'previous_teacher': str(old.teacher_model.iloc[0]),
                  'table': table.to_dict()}
    print(f'{len(both)} users in both pilots (previous teacher: {COMPARISON["previous_teacher"]})')
    print(table.to_string())
else:
    print(f'{old_path} not found: side-by-side skipped')

200 users in both pilots (previous teacher: Qwen/Qwen2.5-7B-Instruct@a09a35458c702b33eeacc393d103063234e8bc28)
                          previous_pilot  qwen35_9b
over_120_words                       0.0        0.0
mentions_digits_like_ids             0.0        2.0
says_all_about_price                13.0        9.0
claims_absence                      17.0        0.0
vague_praise                        31.0        0.0
dislikes_not_mentioned              32.0        6.0
meta_or_reasoning_text               0.0        0.0
mean_words                          42.7       63.4


## 12. Pilot report
Seconds per profile are measured per history-length group and weighted by how many users of each length the full
data has, so the estimate for all 56,017 users is not skewed by the pilot's extra long histories.

In [14]:
full_counts = seq.n_train
timed = res.merge(users[['user_idx', 'n_train']], on='user_idx')
est_secs, strata = 0.0, []
for lo, hi, n in PILOT_STRATA:
    in_s = lambda s: (s >= lo) & ((s <= hi) if hi is not None else True)
    sec = float(timed[in_s(timed.n_train)].seconds.mean())
    n_full = int(in_s(full_counts).sum())
    est_secs += sec * n_full
    strata.append(dict(n_train=f"{lo}-{hi if hi is not None else 'max'}", pilot_users=n, users_in_full_data=n_full,
                       seconds_per_profile=round(sec, 3)))
hours = est_secs / 3600 * (N_USERS_FULL / max(1, len(seq)))
GATES = {'no_failures': len(failed) == 0,
         'no_thinking_text': FLAG_COUNTS['raw_had_think_tag'] == 0 and FLAG_COUNTS['meta_or_reasoning_text'] == 0,
         'at_most_2pct_cut_at_max_tokens': FLAG_COUNTS['hit_max_tokens'] <= 0.02 * len(profiles),
         'no_possible_sensitive_terms': FLAG_COUNTS['possible_sensitive_terms'] == 0,
         'batching_consistent': CONSISTENCY['min_similarity'] >= 0.8}
REPORT = {
    'users': int(len(res)), 'profiles': int(len(profiles)), 'failed': int(len(failed)),
    'failure_rate': round(len(failed) / max(1, len(res)), 4),
    'mean_words': round(float(words.mean()), 1), 'median_words': float(words.median()),
    'seconds_per_profile_pilot_mean': round(float(res.seconds.mean()), 3),
    'by_history_length': strata,
    'estimated_hours_all_56017_users': round(hours, 2),
    'estimated_cost_usd': None if COST_PER_GPU_HOUR_USD is None else round(hours * COST_PER_GPU_HOUR_USD, 2),
    'gpu': GPU_NAME, 'gpu_memory_gb': round(VRAM_GB, 1), 'teacher': TEACHER_TAG, 'teacher_class': TEACHER_CLASS,
    'loaded_as': LOAD_NOTE, 'enable_thinking': ENABLE_THINKING, 'batch_size_start': BATCH_SIZE,
    'flags': FLAG_COUNTS, 'consistency': CONSISTENCY, 'comparison': COMPARISON,
    'automatic_gates': GATES, 'automatic_gates_passed': all(GATES.values()),
    'manual_review_of_review_sample_csv': 'PENDING: read the 20 profiles and record the result in the team notes',
    'prompt_version': PROMPT_VERSION,
}
(OUT_DIR / 'pilot_report.json').write_text(json.dumps(REPORT, indent=2))
print(json.dumps({k: REPORT[k] for k in ('profiles', 'failed', 'mean_words', 'seconds_per_profile_pilot_mean',
                                         'estimated_hours_all_56017_users', 'loaded_as', 'automatic_gates')}, indent=2))
print('Automatic gates passed:', REPORT['automatic_gates_passed'], '(the manual review is still needed)')

{
  "profiles": 200,
  "failed": 0,
  "mean_words": 63.4,
  "seconds_per_profile_pilot_mean": 2.596,
  "estimated_hours_all_56017_users": 35.25,
  "loaded_as": "torch.bfloat16",
  "automatic_gates": {
    "no_failures": true,
    "no_thinking_text": true,
    "at_most_2pct_cut_at_max_tokens": true,
    "no_possible_sensitive_terms": false,
    "batching_consistent": false
  }
}
Automatic gates passed: False (the manual review is still needed)


## 13. Embed the profiles and write the profile files
The teacher is unloaded first. Both encoders use pinned commits; vectors are L2-normalized. E5 gets the
`query: ` prefix (its convention for texts compared with each other). Files follow the agreed format of
`src/data/profile_file.py`.

In [15]:
import gc
del model; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
from sentence_transformers import SentenceTransformer

ENCODER_SPECS = {'minilm': ('sentence-transformers/all-MiniLM-L6-v2', ''),
                 'e5_large': ('intfloat/multilingual-e5-large', 'query: ')}
for k in ENCODER_SPECS:                                                  # local testing only
    if os.environ.get(f'PROFILES_ENCODER_{k.upper()}'):
        ENCODER_SPECS[k] = (os.environ[f'PROFILES_ENCODER_{k.upper()}'], ENCODER_SPECS[k][1])

ENCODER_META = {}
for name in ENCODERS:
    enc_id, prefix = ENCODER_SPECS[name]
    local = Path(enc_id).exists()
    rev = 'local' if local else ENCODER_REVISIONS[name]
    st = SentenceTransformer(enc_id, revision=None if local else rev, device='cuda' if torch.cuda.is_available() else 'cpu')
    emb = st.encode([prefix + t for t in profiles.profile_text], batch_size=64, normalize_embeddings=True,
                    show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
    del st; gc.collect()
    path = OUT_DIR / f'profiles_{name}.npz'
    np.savez(path, user_idx=profiles.user_idx.to_numpy(dtype=np.int64), embeddings=emb, source=np.array('llm_profile'),
             train_items_only=np.array(True))
    ENCODER_META[name] = dict(model=enc_id, revision=rev, prefix=prefix, dim=int(emb.shape[1]), l2_normalized=True,
                              file=path.name, sha256=sha256_file(path))
    print(f'{path.name}: {emb.shape}, {enc_id}@{rev}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (19 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

profiles_e5_large.npz: (200, 1024), intfloat/multilingual-e5-large@3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3


## 14. Check the files (the same rules as `read_profile_file`) and write `profiles_meta.json`
`checksums.json` lists the SHA-256 of every output file, for the later copy to S3.

In [16]:
for name in ENCODERS:
    with np.load(OUT_DIR / f'profiles_{name}.npz', allow_pickle=False) as z:
        assert set(z.files) == {'user_idx', 'embeddings', 'source', 'train_items_only'}
        uid, emb, src, flag = z['user_idx'], z['embeddings'], z['source'], z['train_items_only']
        assert flag.ndim == 0 and flag.dtype == np.bool_ and bool(flag)
        assert src.ndim == 0 and src.dtype.kind == 'U' and str(src) == 'llm_profile'
        assert uid.ndim == 1 and uid.dtype == np.int64 and len(np.unique(uid)) == len(uid) and uid.min() >= 0
        assert uid.max() < N_USERS_FULL
        assert emb.ndim == 2 and emb.shape[0] == len(uid) and emb.dtype == np.float32 and np.isfinite(emb).all()
        assert np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-4)
        assert (uid == profiles.user_idx.to_numpy()).all(), 'npz rows must match user_profiles.parquet'
    print('OK', name, emb.shape)

meta = {
    'mode': 'pilot', 'run_name': RUN_NAME,
    'created_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'teacher_model': TEACHER_MODEL if not IS_LOCAL_MODEL else 'local test model', 'teacher_revision': TEACHER_REV,
    'teacher_main_at_run_time': CURRENT_MAIN, 'teacher_class': TEACHER_CLASS, 'teacher_loaded_as': LOAD_NOTE,
    'enable_thinking': ENABLE_THINKING, 'generation_prompt_tail': GEN_PROMPT_TAIL, 'gpu': GPU_NAME,
    'generation': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS, 'seed': SEED, 'batch_size_start': BATCH_SIZE},
    'prompt_version': PROMPT_VERSION, 'prompt_template': PROMPT_TEMPLATE, 'prompt_sha256': PROMPT_SHA,
    'item_text_fields_kept': list(FIELDS_KEPT), 'max_items_per_user': MAX_ITEMS, 'items_order': 'oldest to newest',
    'inputs': {'data_dir': str(DATA_DIR), 'sha256': INPUT_SHA256, 'manifest_sha256': EXPECTED_MANIFEST_SHA256,
               'columns_read': {'sequences.parquet': ['user_idx', 'train_items', 'n_train'],
                                'item_text.parquet': ['item_idx', 'item_text'],
                                'interactions.parquet': ['user_idx', 'item_idx', 'rating', "split (only 'train' rows used)"]},
               'per_user_input_hash': 'sha256 of {"items": items_used, "ratings": ratings_used}, column input_sha256'},
    'selection_rule': SELECTION_RULE, 'n_users_selected': int(len(users)), 'n_profiles': int(len(profiles)),
    'n_failed': int(len(failed)), 'encoders': ENCODER_META, 'pilot_report': REPORT,
    'versions': {n: _ver(n) for n in ('torch', 'transformers', 'sentence-transformers', 'huggingface-hub', 'accelerate',
                                      'bitsandbytes', 'numpy', 'pandas', 'pyarrow')},
    'python': sys.version.split()[0],
}
(OUT_DIR / 'profiles_meta.json').write_text(json.dumps(meta, indent=2))
files = sorted(p for p in OUT_DIR.rglob('*') if p.is_file() and p.name != 'checksums.json')
(OUT_DIR / 'checksums.json').write_text(json.dumps({str(p.relative_to(OUT_DIR)): sha256_file(p) for p in files}, indent=1))
print('Files in', OUT_DIR, ':', sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))

OK minilm (200, 384)
OK e5_large (200, 1024)
Files in /content/drive/MyDrive/298A/model1_profiles/pilot_v2_qwen35_9b : ['checksums.json', 'comparison_with_pilot_v2.csv', 'consistency_check.csv', 'failed_users.json', 'pilot_report.json', 'profile_flags.csv', 'profiles_e5_large.npz', 'profiles_meta.json', 'profiles_minilm.npz', 'review_sample.csv', 'user_profiles.parquet']


## 15. Finish: what to send to Abhinita, and make sure everything reached Drive

In [17]:
print('For Abhinita (prototype endpoint):')
print(f'  model {TEACHER_MODEL}, commit {TEACHER_REV}, loaded as {LOAD_NOTE}, thinking off (enable_thinking=False)')
print(f'  measured here: {REPORT["seconds_per_profile_pilot_mean"]} s per profile on {GPU_NAME}')
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('All files are saved to Google Drive:', OUT_DIR)
else:
    print('Files are in', OUT_DIR, '- if this is a synced Drive folder, wait until the Drive app shows it is up to date.')

For Abhinita (prototype endpoint):
  model Qwen/Qwen3.5-9B, commit c202236235762e1c871ad0ccb60c8ee5ba337b9a, loaded as torch.bfloat16, thinking off (enable_thinking=False)
  measured here: 2.596 s per profile on NVIDIA L4
All files are saved to Google Drive: /content/drive/MyDrive/298A/model1_profiles/pilot_v2_qwen35_9b
